# HAST rollout: who benefits from better targeting?

This study follows the chronic-mortality and prehistory analyses. The previous figures establish that classifier accuracy can reduce mortality among true chronic applicants, but they pool together people already trapped in the queue and people who arrive after the classifier is introduced.

The question here is therefore:

> When a more accurate HAST-style classifier is introduced, does it reduce chronic mortality for the inherited backlog, for new applicants, or for both?

The model preserves the existing baseline: renewal arrivals, monthly housing releases, a six-month eligibility delay, strict priority for people classified as chronic, and a realistic initial queue generated through naive-FCFS prehistory.

In [ ]:
from dataclasses import dataclass
from functools import cached_property
from math import erf, exp, log, pi, sqrt
from pathlib import Path
import os
import sys

from joblib import Parallel, delayed

# Find the repository's src directory regardless of where Jupyter starts.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    src_dir = candidate / "src"
    if (src_dir / "gi_gi_n_gi_multiclass").is_dir():
        PROJECT_ROOT = candidate
        sys.path.insert(0, str(src_dir))
        break
else:
    raise ImportError("Could not find src/gi_gi_n_gi_multiclass from this notebook location.")

import numpy as np
import pandas as pd
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from gi_gi_n_gi_multiclass.core.engine_multi import run_sim_multi
from gi_gi_n_gi_multiclass.core.prehistory import generate_housing_queue_snapshot
from gi_gi_n_gi_multiclass.core.types_multi import CustomerClass, MultiSimConfig
from gi_gi_n_gi_multiclass.dists.common import Deterministic, Exponential, LogNormal
from gi_gi_n_gi_multiclass.labelling import ConfusionMatrixClassifier
from gi_gi_n_gi_multiclass.outputs.schemas import customers_to_dataframe
from gi_gi_n_gi_multiclass.policies.naive_fcfs import NaiveFCFS
from gi_gi_n_gi_multiclass.policies.priority_fcfs import PriorityFCFS

## Baseline system and study boundary

The study deliberately changes **only classifier accuracy**. Housing supply and the mandatory eligibility delay remain fixed, so that the results answer an attribution question rather than repeat the earlier capacity and wait-time sweeps.

The inherited queue is generated under naive FCFS, before the new classifier exists. At rollout, every unresolved person in that queue is classified using the scenario's classifier, exactly as new applicants are classified at arrival.

In [ ]:
MONTHS_PER_YEAR = 12

ARRIVALS_PER_MONTH = 750
CHRONIC_SHARE = 0.38
CHRONIC_ARRIVAL_RATE = ARRIVALS_PER_MONTH * CHRONIC_SHARE * MONTHS_PER_YEAR
NONCHRONIC_ARRIVAL_RATE = ARRIVALS_PER_MONTH * (1.0 - CHRONIC_SHARE) * MONTHS_PER_YEAR

INITIAL_QUEUE_SIZE = 8_000
MAX_PREHISTORY_YEARS = 25.0

HOUSING_PER_MONTH = 300
HOUSING_RELEASE_INTERVAL = 1.0 / MONTHS_PER_YEAR
FIRST_HOUSING_RELEASE = HOUSING_RELEASE_INTERVAL
INITIAL_HOUSING_STOCK = 0
MANDATORY_WAIT_MONTHS = 6

# Follow post-rollout arrivals for five years and observe all outcomes through year 10.
LAST_ARRIVAL_TIME = 5.0
STUDY_HORIZON_YEARS = 10.0
OBSERVATION_YEARS = np.array([1.0, 3.0, 5.0, 10.0])

# These are scenario accuracies, not yet empirical HAST operating points.
ACCURACY_GRID = np.array([0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90])
CURRENT_ASSESSMENT_ACCURACY = 0.55
HAST_SCENARIO_ACCURACY = 0.85

FIGURE_SEEDS = list(range(1, 11))
N_WORKERS = min(16, os.cpu_count() or 1)
POST_SNAPSHOT_SEED_OFFSET = 1_000_000

FIGURE_SIZE = (7.0, 5.0)
FIGURE_DPI = 200
FONT_SIZE = 13
TITLE_FONT_SIZE = 1.2 * FONT_SIZE
matplotlib.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "axes.titlesize": TITLE_FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
})

FIGURE_DIR = PROJECT_ROOT / "04_homeless_studies" / "figures" / "hast_rollout_cohort_effects"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, filename):
    fig.tight_layout(pad=0.8)
    fig.savefig(FIGURE_DIR / filename, dpi=FIGURE_DPI)

pd.DataFrame(
    [
        ("Initial queue", INITIAL_QUEUE_SIZE, "people"),
        ("New applicants", ARRIVALS_PER_MONTH, "per month"),
        ("True chronic share", CHRONIC_SHARE, "proportion"),
        ("Housing releases", HOUSING_PER_MONTH, "placements per month"),
        ("Mandatory eligibility wait", MANDATORY_WAIT_MONTHS, "months"),
        ("Post-rollout arrivals", LAST_ARRIVAL_TIME, "years"),
        ("Outcome horizon", STUDY_HORIZON_YEARS, "years"),
    ],
    columns=["Parameter", "Value", "Unit"],
)

## Mortality and patience assumptions

As in the previous study, a true chronic person's abandonment event is interpreted as death before housing. The proper Gamma distribution is calibrated so that mortality by 30 months is 4%. Non-chronic people use the existing illustrative exit-time distribution; this is the mechanism through which people may leave the queue after finding another arrangement.

In [ ]:
@dataclass(frozen=True)
class ScaledGammaMortality:
    """Gamma mortality time calibrated to a 4% cumulative probability by 30 months."""

    shape: float = 1.5
    calibration_window_years: float = 30.0 / 12.0
    mortality_by_window: float = 0.04

    @staticmethod
    def _shape_three_halves_cdf(x: float) -> float:
        if x <= 0.0:
            return 0.0
        return erf(sqrt(x)) - (2.0 / sqrt(pi)) * sqrt(x) * exp(-x)

    @cached_property
    def scale_years(self) -> float:
        lower, upper = 0.0, 100.0
        for _ in range(100):
            midpoint = 0.5 * (lower + upper)
            if self._shape_three_halves_cdf(midpoint) < self.mortality_by_window:
                lower = midpoint
            else:
                upper = midpoint
        return self.calibration_window_years / (0.5 * (lower + upper))

    def sample(self, rng: np.random.Generator) -> float:
        return float(rng.gamma(shape=self.shape, scale=self.scale_years))


chronic_mortality = ScaledGammaMortality()
NONCHRONIC_PATIENCE = LogNormal(meanlog=-1.0, sdlog=1.0)

mortality_check = chronic_mortality._shape_three_halves_cdf(
    chronic_mortality.calibration_window_years / chronic_mortality.scale_years
)
nonchronic_exit_by_six_months = 0.5 * (
    1.0 + erf((log(0.5) - NONCHRONIC_PATIENCE.meanlog) / (NONCHRONIC_PATIENCE.sdlog * sqrt(2.0)))
)

assert np.isclose(mortality_check, 0.04)
print(f"Chronic mortality by 30 months: {mortality_check:.1%}")
print(f"Non-chronic exit probability by six months: {nonchronic_exit_by_six_months:.1%}")

## Simulation helpers

The class labels are deliberately asymmetric: sensitivity is five percentage points above the displayed accuracy and specificity is five percentage points below it. Thus the displayed accuracy is balanced accuracy, while the simulation retains the false-positive/false-negative asymmetry used in the preceding mortality study.

Every accuracy scenario for a given seed uses the same prehistory snapshot and the same post-rollout random streams. This paired design reduces Monte Carlo noise when comparing classifiers.

In [ ]:
chronic = CustomerClass(
    name="chronic",
    interarrival=Exponential(rate=CHRONIC_ARRIVAL_RATE),
    service=Deterministic(0.0),
    patience=chronic_mortality,
)
nonchronic = CustomerClass(
    name="nonchronic",
    interarrival=Exponential(rate=NONCHRONIC_ARRIVAL_RATE),
    service=Deterministic(0.0),
    patience=NONCHRONIC_PATIENCE,
)
classes = [chronic, nonchronic]
prehistory_policy = NaiveFCFS()
rollout_policy = PriorityFCFS(priority_order=[0, 1])


def classifier_from_accuracy(accuracy: float) -> ConfusionMatrixClassifier:
    sensitivity = accuracy + 0.05
    specificity = accuracy - 0.05
    if not (0.0 <= sensitivity <= 1.0 and 0.0 <= specificity <= 1.0):
        raise ValueError("Accuracy must keep sensitivity and specificity in [0, 1].")
    return ConfusionMatrixClassifier(
        np.array([
            [sensitivity, 1.0 - sensitivity],
            [1.0 - specificity, specificity],
        ])
    )


def make_config(
    *,
    seed: int,
    classifier_accuracy: float,
    run_years: float,
    initial_queue=None,
    first_housing_release: float = FIRST_HOUSING_RELEASE,
    initial_housing_stock: int = INITIAL_HOUSING_STOCK,
    allocation_policy=rollout_policy,
    classify_initial_queue: bool = False,
) -> MultiSimConfig:
    return MultiSimConfig(
        n_servers=HOUSING_PER_MONTH,
        classes=classes,
        policy=allocation_policy,
        classifier=classifier_from_accuracy(classifier_accuracy),
        seed=seed,
        run_time=run_years,
        housing_mode=True,
        housing_arrival_mode="renewal",
        last_arrival_time=LAST_ARRIVAL_TIME if initial_queue is not None else run_years,
        initial_queue=initial_queue,
        classify_initial_queue=classify_initial_queue,
        housing_release_interval=HOUSING_RELEASE_INTERVAL,
        first_housing_release=first_housing_release,
        initial_housing_stock=initial_housing_stock,
        eligibility_delay=MANDATORY_WAIT_MONTHS / MONTHS_PER_YEAR,
    )


def make_prehistory_snapshot(seed: int):
    config = make_config(
        seed=seed,
        classifier_accuracy=0.80,
        run_years=MAX_PREHISTORY_YEARS,
        allocation_policy=prehistory_policy,
    )
    return generate_housing_queue_snapshot(
        config,
        target_population=INITIAL_QUEUE_SIZE,
        max_prehistory_time=MAX_PREHISTORY_YEARS,
    )

## Cohort definitions and measurement

The study has three mutually exclusive cohorts:

- **Inherited backlog:** people present at the rollout date.
- **Early entrants:** people arriving during the first two rollout years.
- **Later entrants:** people arriving from year 2 through year 5.

Outcomes are reconstructed at years 1, 3, 5, and 10. This lets the notebook distinguish a person who has already been housed, died before housing, or remains unresolved at each observation point, even though the simulation is run once to the full horizon.

In [ ]:
COHORT_ORDER = ["Inherited backlog", "Early entrants (0-2 years)", "Later entrants (2-5 years)"]
COHORT_COLOURS = {
    "Inherited backlog": "#8C2D4B",
    "Early entrants (0-2 years)": "#007C91",
    "Later entrants (2-5 years)": "#D17800",
}


def cohort_labels(customers: pd.DataFrame, initial_queue_count: int) -> pd.Series:
    labels = pd.Series(index=customers.index, dtype="object")
    initial = customers["customer_id"] < initial_queue_count
    labels.loc[initial] = "Inherited backlog"
    incident_time = customers.loc[~initial, "arrival_time"]
    labels.loc[incident_time.index[incident_time < 2.0]] = "Early entrants (0-2 years)"
    labels.loc[incident_time.index[(incident_time >= 2.0) & (incident_time <= LAST_ARRIVAL_TIME + 1e-12)]] = "Later entrants (2-5 years)"
    return labels


def status_at_time(customers: pd.DataFrame, observation_time: float) -> pd.Series:
    """Classify each arrival by its state at an observation time."""
    arrived = customers["arrival_time"] <= observation_time
    housed = arrived & customers["service_start"].notna() & (customers["service_start"] <= observation_time)
    died_before_housing = (
        arrived
        & customers["abandon_time"].notna()
        & (customers["abandon_time"] <= observation_time)
        & (~customers["service_start"].notna() | (customers["service_start"] > customers["abandon_time"]))
    )
    status = pd.Series("Not yet arrived", index=customers.index, dtype="object")
    status.loc[arrived] = "Unresolved"
    status.loc[housed] = "Housed"
    status.loc[died_before_housing] = "Died before housing"
    return status


def summarise_run(accuracy: float, seed: int, snapshot) -> list[dict]:
    result = run_sim_multi(
        make_config(
            seed=seed + POST_SNAPSHOT_SEED_OFFSET,
            classifier_accuracy=accuracy,
            run_years=STUDY_HORIZON_YEARS,
            initial_queue=snapshot.customers,
            first_housing_release=snapshot.time_to_next_housing_release,
            initial_housing_stock=snapshot.available_housing,
            classify_initial_queue=True,
        )
    )
    customers = customers_to_dataframe(result)
    customers["cohort"] = cohort_labels(customers, len(snapshot.customers))
    chronic = customers[customers["true_class_id"] == 0].copy()

    rows = []
    for observation_time in OBSERVATION_YEARS:
        chronic["status"] = status_at_time(chronic, float(observation_time))
        for cohort in COHORT_ORDER:
            group = chronic[chronic["cohort"] == cohort]
            arrived = group[group["arrival_time"] <= observation_time]
            if arrived.empty:
                continue
            status_share = arrived["status"].value_counts(normalize=True)
            rows.append({
                "classifier_accuracy": accuracy,
                "seed": seed,
                "observation_year": observation_time,
                "cohort": cohort,
                "true_chronic_arrived": len(arrived),
                "death_probability": status_share.get("Died before housing", 0.0),
                "housed_probability": status_share.get("Housed", 0.0),
                "unresolved_probability": status_share.get("Unresolved", 0.0),
            })
    return rows


def run_study():
    snapshots = Parallel(n_jobs=N_WORKERS, prefer="processes")(
        delayed(make_prehistory_snapshot)(int(seed)) for seed in FIGURE_SEEDS
    )
    snapshot_by_seed = dict(zip(FIGURE_SEEDS, snapshots))
    scenarios = [
        (float(accuracy), int(seed), snapshot_by_seed[seed])
        for accuracy in ACCURACY_GRID
        for seed in FIGURE_SEEDS
    ]
    nested_rows = Parallel(n_jobs=N_WORKERS, prefer="processes")(
        delayed(summarise_run)(*scenario) for scenario in scenarios
    )
    return pd.DataFrame([row for rows in nested_rows for row in rows]), snapshot_by_seed


cohort_raw, PREHISTORY_SNAPSHOTS = run_study()
replications = cohort_raw.groupby(["classifier_accuracy", "observation_year", "cohort"]).size()
assert replications.eq(len(FIGURE_SEEDS)).all(), "A scenario is missing a replication."

cohort_summary = cohort_raw.groupby(
    ["classifier_accuracy", "observation_year", "cohort"], as_index=False
).agg(
    death_probability=("death_probability", "mean"),
    housed_probability=("housed_probability", "mean"),
    unresolved_probability=("unresolved_probability", "mean"),
    death_probability_se=("death_probability", "sem"),
)

cohort_summary.head()

## Figure 1: mortality by cohort

If the improved classifier mainly helps newly arriving people, the inherited-backlog line will remain comparatively flat. If it reallocates housing towards chronic people who were already waiting, the inherited-backlog line should fall as accuracy improves.

The figure reports the year-10 outcome so that every cohort has a meaningful period of exposure.

In [ ]:
year_10 = cohort_summary[cohort_summary["observation_year"] == STUDY_HORIZON_YEARS]

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
for cohort in COHORT_ORDER:
    data = year_10[year_10["cohort"] == cohort].sort_values("classifier_accuracy")
    ax.plot(
        data["classifier_accuracy"],
        data["death_probability"],
        marker="o",
        linewidth=2.4,
        color=COHORT_COLOURS[cohort],
        label=cohort,
    )
ax.axvline(CURRENT_ASSESSMENT_ACCURACY, color="#303030", linestyle="--", linewidth=1.2, label="Current-assessment scenario")
ax.axvline(HAST_SCENARIO_ACCURACY, color="#303030", linestyle=":", linewidth=1.5, label="Improved-HAST scenario")
ax.set(
    xlabel="Classifier accuracy (balanced accuracy)",
    ylabel="P(death before housing | true chronic)",
    title=f"Chronic mortality by rollout cohort after {STUDY_HORIZON_YEARS:g} years",
)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=1))
ax.grid(axis="y", color="#D9D9D9", linewidth=0.7)
ax.legend(frameon=True, fancybox=False, framealpha=1.0, edgecolor="#303030")
save_figure(fig, "figure_1_cohort_mortality_vs_accuracy.png")
plt.show()

## Figure 2: disposition of the inherited chronic backlog

This figure keeps attention on the people present when the policy changes. At each accuracy, the stacked bar gives their year-10 disposition: housed, died before housing, or still unresolved. It therefore separates a reduction in mortality from merely deferring the outcome.

In [ ]:
backlog_year_10 = year_10[year_10["cohort"] == "Inherited backlog"].sort_values("classifier_accuracy")

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
accuracy = backlog_year_10["classifier_accuracy"]
housed = backlog_year_10["housed_probability"]
died = backlog_year_10["death_probability"]
unresolved = backlog_year_10["unresolved_probability"]

ax.bar(accuracy, housed, width=0.035, color="#4C9AC0", label="Housed")
ax.bar(accuracy, died, width=0.035, bottom=housed, color="#B54A4A", label="Died before housing")
ax.bar(accuracy, unresolved, width=0.035, bottom=housed + died, color="#A8A8A8", label="Still unresolved")
ax.axvline(CURRENT_ASSESSMENT_ACCURACY, color="#303030", linestyle="--", linewidth=1.2)
ax.axvline(HAST_SCENARIO_ACCURACY, color="#303030", linestyle=":", linewidth=1.5)
ax.set(
    xlabel="Classifier accuracy (balanced accuracy)",
    ylabel="Share of inherited true chronic backlog",
    title=f"Disposition of inherited chronic people after {STUDY_HORIZON_YEARS:g} years",
    ylim=(0, 1),
)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.grid(axis="y", color="#D9D9D9", linewidth=0.7)
ax.legend(frameon=True, fancybox=False, framealpha=1.0, edgecolor="#303030")
save_figure(fig, "figure_2_inherited_backlog_disposition.png")
plt.show()

## Figure 3: when mortality gains appear

This paired comparison fixes the low-accuracy scenario as the reference and asks how much chronic mortality falls under the improved-HAST scenario at each observation time. Because each pair uses the same prehistory and post-rollout random streams, differences are attributable to the classifier scenario rather than a different realised queue.

In [ ]:
paired = cohort_raw[cohort_raw["classifier_accuracy"].isin([CURRENT_ASSESSMENT_ACCURACY, HAST_SCENARIO_ACCURACY])].pivot(
    index=["seed", "observation_year", "cohort"],
    columns="classifier_accuracy",
    values="death_probability",
).reset_index()
paired["mortality_reduction"] = paired[CURRENT_ASSESSMENT_ACCURACY] - paired[HAST_SCENARIO_ACCURACY]
paired_summary = paired.groupby(["observation_year", "cohort"], as_index=False).agg(
    mortality_reduction=("mortality_reduction", "mean"),
    mortality_reduction_se=("mortality_reduction", "sem"),
)

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
for cohort in COHORT_ORDER:
    data = paired_summary[paired_summary["cohort"] == cohort].sort_values("observation_year")
    ax.plot(
        data["observation_year"],
        data["mortality_reduction"],
        marker="o",
        linewidth=2.4,
        color=COHORT_COLOURS[cohort],
        label=cohort,
    )
    ax.fill_between(
        data["observation_year"],
        data["mortality_reduction"] - 1.96 * data["mortality_reduction_se"],
        data["mortality_reduction"] + 1.96 * data["mortality_reduction_se"],
        color=COHORT_COLOURS[cohort],
        alpha=0.14,
    )
ax.axhline(0, color="#303030", linewidth=1.0)
ax.set(
    xlabel="Years after HAST rollout",
    ylabel="Reduction in P(death before housing | true chronic)",
    title="Paired chronic-mortality reduction from improved targeting",
    xticks=OBSERVATION_YEARS,
)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=1))
ax.grid(axis="y", color="#D9D9D9", linewidth=0.7)
ax.legend(frameon=True, fancybox=False, framealpha=1.0, edgecolor="#303030")
save_figure(fig, "figure_3_timing_of_mortality_reduction.png")
plt.show()

## Compact results table

The table supports reporting by showing all three dispositions at the end of the horizon. The key comparison is whether the inherited backlog benefits to the same extent as the people entering after rollout.

In [ ]:
final_summary = cohort_summary[
    cohort_summary["observation_year"] == STUDY_HORIZON_YEARS
].pivot(
    index="cohort",
    columns="classifier_accuracy",
    values=["death_probability", "housed_probability", "unresolved_probability"],
)
display(final_summary.round(3))

paired_summary.round(4)

## Interpretation guide

- A fall in inherited-backlog mortality means the rollout is helping people already waiting, not merely improving outcomes for future entrants.
- A fall only for incident cohorts means improved targeting is valuable, but does not by itself remedy historical queue disadvantage.
- A mortality reduction accompanied by a rising unresolved share should be interpreted as delayed harm rather than a complete resolution.
- These accuracy values remain scenario inputs. Once an empirical HAST threshold and confusion matrix are available, they can replace the grid without changing the study design.